# LangChain L5 — Level 4 — Structured output
OpsPilot v3 answers in prose. A ticketing system cannot route on "it seems the customer wants
a refund". It needs:

```json
{"intent": "billing", "customer_id": "C002", "priority": "high", "department": "finance"}
```

```text
Natural language  ->  Model  ->  validated object  ->  business logic
```

With `response_format`, `create_agent()` makes the model end its run by producing an object that
matches a Pydantic schema. `ToolStrategy` implements this as one more tool call (the schema is
the tool), which works on every tool-calling model; `ProviderStrategy` uses a provider's native
JSON-schema mode when available. The result appears under `result["structured_response"]`.

### Step 1 — Define the schema and the agent

`Literal` fields constrain values; `Field(description=...)` tells the model what each field means.
Validation runs on the model's output, so a bad value becomes a retry, not a bad database row.

In [ ]:
from typing import Literal                                      # Python standard library
from langchain.agents.structured_output import ToolStrategy     # LangChain: "the schema is a tool" strategy

class SupportTicket(BaseModel):                                 # ours, on Pydantic's BaseModel
    """A classified support ticket ready for routing."""
    intent: Literal["billing", "shipping", "technical", "general"] = Field(description="What the customer needs.")
    customer_id: str = Field(description="Customer id if mentioned, otherwise 'unknown'.")
    priority: Literal["low", "medium", "high"] = Field(description="high for money already lost or outages.")
    department: Literal["finance", "logistics", "support"] = Field(description="Team that should own the ticket.")

classifier = create_agent(
    model=model,
    tools=[get_customer],                                   # it may still look things up first
    system_prompt="You classify incoming support messages for Meridian Supply Co. Look up the customer if an id is given.",
    response_format=ToolStrategy(SupportTicket),                # LangChain: end the run with a validated object
)

result = classifier.invoke({"messages": [{"role": "user", "content": "Customer C002 here. My payment for the server rack went through twice, please fix this urgently."}]})   # LangGraph
ticket = result["structured_response"]                          # LangChain: the validated SupportTicket
print("type      :", type(ticket).__name__)
print("ticket    :", ticket)
print("as dict   :", ticket.model_dump())                        # Pydantic: object -> dict

### Step 2 — Structured output feeds ordinary code

Once the answer is an object, routing is plain Python: no regexes over prose, no guessing.

In [ ]:
ROUTING = {"finance": "finance-queue@meridian", "logistics": "ops-queue@meridian", "support": "help-queue@meridian"}

def route_ticket(ticket: SupportTicket) -> str:   # ours: plain business logic
    queue = ROUTING[ticket.department]
    flag = " [ESCALATE]" if ticket.priority == "high" else ""
    return f"ticket for {ticket.customer_id} -> {queue}{flag}"

print(route_ticket(ticket))

### Recap

- **Problem seen:** prose answers cannot be routed, stored or validated.
- **Layer added:** `response_format=ToolStrategy(Schema)` and `result['structured_response']`.
- **Evidence:** a validated `SupportTicket` object drove a routing function with no text parsing.